# Colebrookova jednadžba: predvidi → izračunaj → provjeri

Moodyjev dijagram nije samo slika: u turbulentnom području svaka njegova krivulja zadovoljava implicitnu Colebrookovu jednadžbu za Darcyjev faktor trenja \(\lambda\).

## Predvidi

Prije iteracije procijeni:

1. Povećava li relativna hrapavost \(\varepsilon/D\) faktor trenja?
2. Hoće li dvije razumne početne pretpostavke završiti na istom korijenu?
3. Zašto Colebrookovu jednadžbu ne treba primjenjivati na laminarni tok?

Prijelazno područje \(2300\lesssim Re\lesssim4000\) ovdje namjerno ne interpoliramo: režim može biti nestabilan i jedna glatka formula skriva tu neizvjesnost.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

plt.rcParams.update({"figure.dpi": 110, "font.size": 10})

def colebrook_residual(lam, Re, rel_roughness):
    return 1/np.sqrt(lam) + 2*np.log10(rel_roughness/3.7 + 2.51/(Re*np.sqrt(lam)))

def colebrook_iter(Re, rel_roughness, x0=7.0, tol=1e-12, max_iter=100):
    # Fiksna točka u varijabli x=1/sqrt(lambda), uz povijest iteracija.
    if Re <= 4000 or rel_roughness < 0:
        raise ValueError("Colebrookov račun ovdje vrijedi za Re > 4000 i ε/D ≥ 0.")
    x = float(x0)
    history = []
    for iteration in range(max_iter + 1):
        lam = 1/x**2
        residual = colebrook_residual(lam, Re, rel_roughness)
        history.append((iteration, lam, residual))
        x_new = -2*np.log10(rel_roughness/3.7 + 2.51*x/Re)
        if abs(x_new-x) < tol:
            x = x_new
            lam = 1/x**2
            history.append((iteration+1, lam, colebrook_residual(lam, Re, rel_roughness)))
            return lam, np.asarray(history)
        x = x_new
    raise RuntimeError("Iteracija nije konvergirala unutar zadanog broja koraka.")

Re0, rr0 = 1.0e5, 1.0e-4
lam0, history = colebrook_iter(Re0, rr0, x0=5.0)
print(f"lambda = {lam0:.8f}; iteracija = {len(history)-1}; završni rezidual = {history[-1,2]:.3e}")
print(" i       lambda       rezidual")
for row in history:
    print(f"{int(row[0]):2d}   {row[1]:.9f}   {row[2]: .3e}")


## Izračunaj: konvergencija i osjetljivost na hrapavost

Rezidual je lijeva strana implicitne jednadžbe i mora težiti nuli. Zatim za isti Reynoldsov broj mijenjamo \(\varepsilon/D\) kroz četiri reda veličine. To je numerička verzija horizontalnog presjeka Moodyjeva dijagrama.


In [ ]:
lam_other_start, history_other = colebrook_iter(Re0, rr0, x0=12.0)
roughness = np.logspace(-6, -2, 70)
lam_rough = np.array([colebrook_iter(Re0, rr)[0] for rr in roughness])

def friction_factor(Re, rel_roughness=0.0):
    if Re < 2300:
        return 64/Re
    if Re <= 4000:
        raise ValueError("Prijelazno područje nema jedinstvenu vrijednost u ovom modelu.")
    return colebrook_iter(Re, rel_roughness)[0]

lam_laminar = friction_factor(1200, rr0)
print(f"Laminarno, Re=1200: lambda = {lam_laminar:.6f}")
print(f"Turbulentno: promjena λ od {lam_rough[0]:.5f} do {lam_rough[-1]:.5f}")


## Provjeri

Neovisne provjere su: zatvaranje implicitne jednadžbe, neovisnost korijena o početnoj pretpostavci i analitički laminarni granični slučaj \(\lambda=64/Re\).


In [ ]:
assert abs(colebrook_residual(lam0, Re0, rr0)) < 1e-10
assert np.isclose(lam_other_start, lam0, rtol=1e-11)
assert np.isclose(lam_laminar, 64/1200, rtol=1e-14)
assert np.all(np.diff(lam_rough) > 0)

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
axes[0].semilogy(history[:,0], np.maximum(np.abs(history[:,2]), 1e-16), "o-", color="#b43c35")
axes[0].set(xlabel="iteracija", ylabel="|Colebrookov rezidual|", title="Povijest konvergencije")
axes[1].semilogx(roughness, lam_rough, color="#256d85", lw=2)
axes[1].scatter([rr0], [lam0], color="#b43c35", zorder=3)
axes[1].set(xlabel=r"relativna hrapavost $\varepsilon/D$", ylabel=r"Darcyjev $\lambda$", title=f"Osjetljivost pri Re={Re0:.0e}")
for ax in axes: ax.grid(True, which="both", ls=":", alpha=.45)
plt.tight_layout(); plt.show()


## Protumači

Promijeni \(Re\) za ±10 % i usporedi taj učinak s udvostručenjem hrapavosti. Kao kriterij završetka ne koristi samo malu promjenu \(\lambda\): uvijek izvijesti i rezidual izvorne Colebrookove jednadžbe.
